# NN based NGPS Localization

In [1]:
from pathlib import Path

if Path.cwd().name != "LightGlue":
    !git clone --quiet https://github.com/snktshrma272/LightGlue
    %cd LightGlue
    !pip install --progress-bar off --quiet -e .

from lightglue import LightGlue, SuperPoint, DISK
from lightglue.utils import load_image, rbd
from lightglue import viz2d
import torch
import cv2 as cv
from PIL import Image
import numpy

torch.set_grad_enabled(False)
images = Path("assets")

/content/LightGlue
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for lightglue (pyproject.toml) ... done


/content/LightGlue/lightglue/lightglue.py:24: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

extractor = SuperPoint(max_num_keypoints=2048).eval().to(device)
matcher = LightGlue(features="superpoint").eval().to(device)

Downloading: "https://github.com/cvg/LightGlue/releases/download/v0.1_arxiv/superpoint_v1.pth" to /root/.cache/torch/hub/checkpoints/superpoint_v1.pth
100%|██████████| 4.96M/4.96M [00:00<00:00, 335MB/s]
Downloading: "https://github.com/cvg/LightGlue/releases/download/v0.1_arxiv/superpoint_lightglue.pth" to /root/.cache/torch/hub/checkpoints/superpoint_lightglue_v0-1_arxiv.pth
100%|██████████| 45.3M/45.3M [00:01<00:00, 32.8MB/s]


In [3]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
from google.colab.patches import cv2_imshow

video_path = '/content/drive/My Drive/full_video.mp4'
image_path = '/content/drive/My Drive/satellite_image.tif'

video = cv.VideoCapture(video_path)
if not video.isOpened():
    print(f"Error: Could not open video file at {video_path}")

image = cv.imread(image_path)
if image is None:
    print(f"Error: Could not open image file at {image_path}")

In [43]:
import matplotlib.pyplot as plt
import numpy as np
def plot_side_by_side(img1, img2, title1="Image 0", title2="Image 1", save_path=None):

    img1_np = img1.cpu().permute(1, 2, 0).numpy()
    img2_np = img2.cpu().permute(1, 2, 0).numpy()

    fig, axs = plt.subplots(1, 2, figsize=(10, 5))
    axs[0].imshow(img1_np)
    axs[0].set_title(title1)
    axs[0].axis('off')

    axs[1].imshow(img2_np)
    axs[1].set_title(title2)
    axs[1].axis('off')

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path)
        print(f"Saved plot to {save_path}")
    else:
        plt.show()

    plt.close()


In [44]:
h = 0
w = 0
def kernel_show(center_row, center_col):
    global h, w
    kernel_size = 1000
    im = Image.open(image_path)
    imarray = np.array(im)
    h, w, c = imarray.shape

    if center_row == 0 and center_col == 0:
        center_row = h // 2
        center_col = w // 2

    margin = kernel_size // 2
    min_row, max_row = margin, h - margin
    min_col, max_col = margin, w - margin

    center_row = np.clip(center_row, min_row, max_row)
    center_col = np.clip(center_col, min_col, max_col)

    imarraySeg = imarray[center_row - margin : center_row + margin,
                         center_col - margin : center_col + margin,
                         :]

    return imarraySeg


def map_kernel_img(x,y,base_x,base_y):
  if base_x == 0 and base_y == 0:
    base_x = h//2
    base_y = w//2
  return base_x+x, base_y+y

In [ ]:
import torchvision.transforms as transforms
from lightglue.utils import load_image_arr
import gc

cap = cv.VideoCapture(video_path)
# cap.set(cv.CAP_PROP_POS_FRAMES, 2000)
n = 0
x,y = [0,0]
base_x , base_y = [0,0]
while (cap.isOpened()):
  base_x,base_y = map_kernel_img(y,x,base_x,base_y)
  img_kl = kernel_show(base_x,base_y)

  print(x,y,base_x,base_y)
  image0 = load_image_arr(img_kl)
  feats0 = extractor.extract(image0.to(device))
  ret, frame = cap.read()
  hh, ww = frame.shape[:2]
  if ret == True:
    # print(image0.ndim)
    if frame.ndim == 3:
      frame = frame.transpose((2,0,1))
    elif frame.ndim == 2:
      frame = frame[None]

    image1 = torch.tensor(frame / 255.0, dtype = torch.float)
    feats1 = extractor.extract(image1.to(device))
    matches01 = matcher({"image0": feats0, "image1": feats1})
    feats0, feats1, matches01 = [rbd(x) for x in [feats0, feats1, matches01]]  # remove batch dimension
    kpts0, kpts1, matches = feats0["keypoints"], feats1["keypoints"], matches01["matches"]
    m_kpts0, m_kpts1 = kpts0[matches[..., 0]], kpts1[matches[..., 1]]

    M, mask = cv.findHomography(m_kpts1.cpu().numpy(), m_kpts0.cpu().numpy(), cv.RANSAC, 5.0)
    # pts = numpy.float32([[0,0],[0,frame.shape[1]],[frame.shape[0], frame.shape[1]],[frame.shape[0],0]]).reshape(-1,1,2)

    pts = np.float32([[0, 0], [ww, 0], [ww, hh], [0, hh]]).reshape(-1, 1, 2)

    dst = cv.perspectiveTransform(pts,M)

    print("dst: ", dst, "src: ", img_kl.shape, "m: ", M, "pts: ", pts)
    imgF = cv.polylines(img_kl,[numpy.int32(dst)],True,255,30, cv.LINE_AA)
    x, y = numpy.mean(dst, axis=0).astype(int)[0]



    imgF = cv.circle(imgF, (x,y), 10, (0,0,255), -1)
    x,y = [x-500, y-500]
    transform = transforms.ToTensor()
    imgTorch = transform(imgF)




    # axes = viz2d.plot_images([imgTorch, image1])
    # # viz2d.plot_matches(m_kpts0, m_kpts1, color="lime", lw=0.2)
    # # viz2d.add_text(0, f'Stop after {matches01["stop"]} layers', fs=20)
    # # print("done")
    # viz2d.save_plot(f'../tt{n}.png')

    plot_side_by_side(imgTorch, image1.cpu(), save_path=f'../tt{n}.png')

    torch.cuda.empty_cache()
     # kpc0, kpc1 = viz2d.cm_prune(matches01["prune0"]), viz2d.cm_prune(matches01["prune1"])
      # viz2d.plot_images([image0, image1])
      # viz2d.plot_keypoints([kpts0, kpts1], colors=[kpc0, kpc1], ps=10)
      # plt.show()
    # print("lalala")
    n+=1
    gc.collect()

In [50]:
import cv2 as cv
img = []
for i in range(439):
  img.append(cv.imread(f"tt{i}" + ".png"))

h,w,d = img[1].shape
video = cv.VideoWriter('fin.mp4', cv.VideoWriter_fourcc(*'mp4v'), 30, (w,h))
for i in range(439):
  video.write(img[i])
video.release()

In [49]:
%cd ..

/content
